<p align="center">
  <a href="https://github.com/wavekat/wavekat-lab">
    <img src="https://github.com/wavekat/wavekat-brand/raw/main/assets/banners/wavekat-lab-narrow.svg" alt="WaveKat Lab">
  </a>
</p>

# Smart-Turn — zero-shot upstream baseline

Score `pipecat-ai/smart-turn-v3` ONNX **directly** on the zh test set,
without any zh fine-tuning. This is the floor: any zh-specific model
we train has to beat this number to justify its existence.

The upstream model was trained on ~270k English turn-taking samples,
so this also tells us how well that prior transfers to Chinese
zero-shot — which informs whether warm-starting from these weights
(`02_d_train_warmstart.ipynb`) is worth pursuing.

Pipeline:

1. Download the ONNX checkpoint from HF (`pipecat-ai/smart-turn-v3`).
2. Load the same `smart-turn-zh` Parquet snapshot the training
   notebooks use.
3. Run `smart_turn.score_onnx` on `ds["test"]`.
4. Report F1 / precision / recall / AP at thr=0.5 and at the
   F1-best threshold.

In [ ]:
from pathlib import Path

DATASET_NAME = "smart-turn-zh"
ONNX_REPO = "pipecat-ai/smart-turn-v3"
ONNX_FILE = "smart-turn-v3.0.onnx"  # the original v3 release; v3.1/v3.2 are CPU/GPU-optimized variants

# Upstream ONNX expects whisper-tiny preprocessing (not ours). Match it.
FE_SOURCE = "openai/whisper-tiny"
TARGET_SR = 16_000
CHUNK_LENGTH = 8

EXPORT_DIR = Path(f"../../datasets/{DATASET_NAME}").resolve()
ONNX_DIR = Path(f"../../checkpoints/upstream/{ONNX_REPO.split('/')[-1]}").resolve()
ONNX_DIR.mkdir(parents=True, exist_ok=True)

assert EXPORT_DIR.exists(), (
    f"{EXPORT_DIR.name} not found. Re-run `wk exports adapt smart-turn --out {EXPORT_DIR}`."
)

print("dataset    :", DATASET_NAME)
print("onnx repo  :", ONNX_REPO)
print("onnx file  :", ONNX_FILE)
print("fe source  :", FE_SOURCE)
print("✅ config ready")

In [ ]:
from huggingface_hub import hf_hub_download

onnx_path = Path(hf_hub_download(
    repo_id=ONNX_REPO,
    filename=ONNX_FILE,
    local_dir=str(ONNX_DIR),
))
size_mb = onnx_path.stat().st_size / 1e6
print(f"onnx file  : {onnx_path.name}  ({size_mb:.1f} MB)")
print("✅ onnx checkpoint ready")

In [ ]:
from datasets import load_dataset, Audio, disable_progress_bars

disable_progress_bars()

data_files = {
    split: str(EXPORT_DIR / f"{split}.parquet")
    for split in ("train", "validation", "test")
    if (EXPORT_DIR / f"{split}.parquet").exists()
}
assert "test" in data_files, "test.parquet missing — adapt step did not produce it."

ds = load_dataset("parquet", data_files=data_files)
ds = ds.cast_column("audio", Audio(sampling_rate=TARGET_SR))
print(ds)
print("✅ dataset loaded")

In [ ]:
from smart_turn import score_onnx

result_default = score_onnx(
    onnx_path=onnx_path,
    feature_extractor_source=FE_SOURCE,
    hf_test_split=ds["test"],
    target_sr=TARGET_SR,
    chunk_length=CHUNK_LENGTH,
    threshold=0.5,
)

print(f"threshold       : {result_default['threshold']:.3f}")
print(f"accuracy        : {result_default['accuracy']:.4f}")
print(f"precision       : {result_default['precision']:.4f}")
print(f"recall          : {result_default['recall']:.4f}")
print(f"f1              : {result_default['f1']:.4f}")
print(f"average precision: {result_default['average_precision']:.4f}")
print("✅ zero-shot scored at thr=0.5")

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score
from smart_turn import THRESHOLDS

probs = result_default["probs"]
labels = result_default["labels"]
f1s = np.array([
    f1_score(labels, (probs > t).astype(int), zero_division=0) for t in THRESHOLDS
])
best = int(np.argmax(f1s))
best_thr = float(THRESHOLDS[best])
best_f1 = float(f1s[best])
best_p = precision_score(labels, (probs > best_thr).astype(int), zero_division=0)
best_r = recall_score(labels, (probs > best_thr).astype(int), zero_division=0)

print(f"best threshold : {best_thr:.3f}")
print(f"  precision    : {best_p:.4f}")
print(f"  recall       : {best_r:.4f}")
print(f"  f1           : {best_f1:.4f}")
print(f"  delta vs 0.5 : {best_f1 - result_default['f1']:+.4f}")
print("✅ threshold-swept zero-shot reported")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

prec, rec, _ = precision_recall_curve(labels, probs)
ap = result_default["average_precision"]

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(rec, prec, color="C0", label=f"upstream v3 (AP={ap:.3f})")
ax.scatter([best_r], [best_p], color="C3", zorder=5,
           label=f"thr={best_thr:.2f}  F1={best_f1:.3f}")
ax.set_xlabel("recall")
ax.set_ylabel("precision")
ax.set_title("Zero-shot upstream on smart-turn-zh test")
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.3)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

print("✅ pr curve plotted")